In [1]:
# Cell 1 - install dependencies
# Run this cell in a fresh Google Colab runtime.
# After it finishes, use Runtime -> Restart session, then run the remaining cells.

import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

%pip install -q --upgrade pip

# Keep the Colab PyTorch/CUDA installation already provided by the runtime.
# Do not downgrade NumPy, protobuf, torch, or CUDA packages in this runtime.
# Install the COMET package without letting its older metadata replace
# the newer Colab runtime packages.
%pip install -q --upgrade --prefer-binary \
    'transformers==4.46.3' \
    'torchmetrics==0.10.3' \
    'pytorch-lightning==2.4.0' \
    'sentencepiece==0.2.0' \
    'sacrebleu>=2.0.0,<3.0.0' \
    'entmax>=1.1,<2.0' \
    'jsonargparse==3.13.1' \
    'scipy' \
    'huggingface-hub<1.0' \
    'setuptools<81'

%pip install -q --upgrade --no-deps 'unbabel-comet==2.2.7'

print('Installation complete.')
print('Please restart the Colab runtime before running Cell 2.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for sentencepiece (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for sentencepiece
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> sentencepiece
Installation complete.
Please restart the Colab runtime before running Cell 2.


In [3]:
%pip install -q --upgrade --prefer-binary \
    "pytorch-lightning==2.4.0" \
    "torchmetrics==0.10.3" \
    "transformers==4.46.3"

In [1]:
# Cell 2 - verify the installed environment
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import numpy as np
import pandas as pd
import torch
import google.protobuf
import transformers

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('protobuf:', google.protobuf.__version__)
print('transformers:', transformers.__version__)

from comet import download_model, load_from_checkpoint
print('COMET imported successfully')

numpy: 2.1.3
pandas: 2.2.3
torch: 2.11.0+cu130
cuda available: True
protobuf: 6.33.6
transformers: 4.46.3


/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


COMET imported successfully


In [2]:
# Cell 3 - COMET imports
from comet import download_model, load_from_checkpoint

In [ ]:
# Cell 4 - optional Hugging Face login
# The COMET-QE model is public, so login is normally not required.
# If Hugging Face asks for authentication in your runtime, run:

In [4]:
from huggingface_hub import login
login()

In [5]:
# Cell 5 - load data and run COMET-QE scoring
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
import pandas as pd
import torch
from comet import download_model, load_from_checkpoint

CN_SOURCE_PATH = '/content/drive/MyDrive/conversation_cn.json'
EN_TRANSLATED_PATH = '/content/drive/MyDrive/conversation_cn_translated_en_deepseek_v4flash.json'

OUTPUT_CSV = '/content/drive/MyDrive/comet/comet_qe_translation_quality_scores.csv'
OUTPUT_OVERALL_SUMMARY = '/content/drive/MyDrive/comet/comet_qe_overall_summary.csv'
OUTPUT_INTENT_SUMMARY = '/content/drive/MyDrive/comet/comet_qe_by_intent_summary.csv'

MODEL_NAME = 'Unbabel/wmt22-cometkiwi-da'
BATCH_SIZE = 8

# Intents to exclude from per-intent summary
EXCLUDE_INTENTS = {'Inquire'}

# Create the output folder if it does not already exist.
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

with open(CN_SOURCE_PATH, 'r', encoding='utf-8') as f:
    cn_data = json.load(f)

with open(EN_TRANSLATED_PATH, 'r', encoding='utf-8') as f:
    en_data = json.load(f)

rows = []

for conv_idx, (cn_conv, en_conv) in enumerate(zip(cn_data, en_data)):
    cn_turns = cn_conv.get('conversation', [])
    en_turns = en_conv.get('conversation', [])

    for turn_idx, (cn_turn, en_turn) in enumerate(zip(cn_turns, en_turns)):
        src = str(cn_turn.get('content', '')).strip()
        mt = str(en_turn.get('content', '')).strip()
        intent = cn_turn.get('intent')

        # Keep labeled user turns only
        if src and mt and intent is not None:
            rows.append({
                'conv_idx': conv_idx,
                'turn_idx': turn_idx,
                'role': cn_turn.get('role'),
                'intent': intent,
                'src': src,
                'mt': mt,
            })

print(f'Total labeled user turns to score: {len(rows)}')

comet_inputs = [{'src': r['src'], 'mt': r['mt']} for r in rows]

print('Downloading/loading COMET-QE model...')
model_path = download_model(MODEL_NAME)
model = load_from_checkpoint(model_path)

gpus = 1 if torch.cuda.is_available() else 0
print('CUDA available:', torch.cuda.is_available())
print('Using GPUs:', gpus)

print('Running COMET-QE scoring...')
pred = model.predict(
    comet_inputs,
    batch_size=BATCH_SIZE,
    gpus=gpus,
)

for r, s in zip(rows, pred.scores):
    r['comet_qe_score'] = float(s)

score_df = pd.DataFrame(rows)
score_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved full scores: {OUTPUT_CSV}')

# Overall summary - all labeled user turns
overall_summary = pd.DataFrame({
    'count': [len(score_df)],
    'mean': [score_df['comet_qe_score'].mean()],
    'std': [score_df['comet_qe_score'].std()],
    'min': [score_df['comet_qe_score'].min()],
    'median': [score_df['comet_qe_score'].median()],
    'max': [score_df['comet_qe_score'].max()],
}, index=['Overall'])

overall_summary.to_csv(OUTPUT_OVERALL_SUMMARY)

# Per-intent summary - exclude Inquire
intent_df = score_df[~score_df['intent'].isin(EXCLUDE_INTENTS)]

intent_summary = (
    intent_df.groupby('intent')['comet_qe_score']
    .agg(['count', 'mean', 'std', 'min', 'median', 'max'])
    .sort_values('mean', ascending=False)
)

intent_summary.to_csv(OUTPUT_INTENT_SUMMARY)

print('\nCOMET-QE overall summary (all labeled user turns):')
display(overall_summary)

print('\nCOMET-QE by intent (Inquire excluded):')
display(intent_summary)

print(f'\nSaved overall summary: {OUTPUT_OVERALL_SUMMARY}')
print(f'Saved intent summary: {OUTPUT_INTENT_SUMMARY}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total labeled user turns to score: 3056
Downloading/loading COMET-QE model...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

hparams.yaml:   0%|          | 0.00/716 [00:00<?, ?B/s]

...


Saved full scores: /content/drive/MyDrive/comet/comet_qe_translation_quality_scores.csv

COMET-QE overall summary (all labeled user turns):


,count,mean,std,min,median,max
Overall,3056,0.810412,0.060406,0.36142,0.824321,0.879834



COMET-QE by intent (Inquire excluded):


,count,mean,std,min,median,max
intent,,,,,,
Chitchat,275,0.851531,0.025057,0.644862,0.857624,0.872094
Interpret,1590,0.812218,0.052368,0.364662,0.823190,0.879834
Reveal,975,0.803431,0.067898,0.395145,0.821092,0.877329
Revise,215,0.776105,0.079627,0.361420,0.797706,0.871575



Saved overall summary: /content/drive/MyDrive/comet/comet_qe_overall_summary.csv
Saved intent summary: /content/drive/MyDrive/comet/comet_qe_by_intent_summary.csv


In [6]:
import pandas as pd

reported = pd.DataFrame([
    ["Chitchat", "0.852 ± 0.025"],
    ["Interpret", "0.812 ± 0.052"],
    ["Reveal", "0.803 ± 0.068"],
    ["Revise", "0.776 ± 0.080"],
    ["Overall", "0.810 ± 0.060"],
], columns=["Intent", "COMET-QE (Mean ± Std)"])

reported.to_csv(
    "/content/drive/MyDrive/comet/comet_qe_summary_reported_in_manuscript.csv",
    index=False,
    encoding="utf-8-sig",
)

reported

,Intent,COMET-QE (Mean ± Std)
0,Chitchat,0.852 ± 0.025
1,Interpret,0.812 ± 0.052
2,Reveal,0.803 ± 0.068
3,Revise,0.776 ± 0.080
4,Overall,0.810 ± 0.060
